# ML-07 — Baseline Action Score and Top-10 Review

## Refresh / Content Opportunity Scoring

This is a transparent, rule-based baseline for a content editor. It ranks pseudonymized pages for **human review**, not automatic editing. The starter snapshot has no future outcome, so `trend_direction == 'down'` is used only as a current-state audit proxy; it is never used in the score.

## 1. Check two signals, then state the rule

**Rule in plain words:** Recommend a human content refresh review when a page has not been updated for at least 180 days and still has at least 500 search impressions in the trailing 90 days. Among qualifying pages, higher-impression and more-stale pages are reviewed first.

The two signals are `days_since_last_update` (a refresh-flag signal) and `impressions_90d` (a visibility/volume signal). I audit their directional relationship with the temporary decline proxy. This does not make the proxy a feature or prove that a refresh causes recovery.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists())
data_path = repo_root / 'data/raw/content_refresh_anonymized.csv'
output_dir = repo_root / 'work/outputs'
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)
# Audit-only proxy: derived from trend_direction, and excluded from all score inputs.
df['decline_proxy_rule'] = df['trend_direction'].eq('down')
print(f'Loaded {len(df):,} pseudonymized content items.')

Loaded 30,000 pseudonymized content items.


In [2]:
def bucket_audit(frame, column, bins, labels):
    bucket = pd.cut(frame[column], bins=bins, labels=labels, include_lowest=True, right=False)
    return (frame.assign(bucket=bucket)
            .groupby('bucket', observed=False)
            .agg(n=('decline_proxy_rule', 'size'),
                 declining_rate=('decline_proxy_rule', 'mean'),
                 median_impressions=('impressions_90d', 'median'))
            .assign(declining_rate=lambda x: x['declining_rate'].map(lambda v: f'{v:.1%}')))

freshness_audit = bucket_audit(
    df, 'days_since_last_update',
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=['0–29 days', '30–89 days', '90–179 days', '180–364 days', '365+ days']
)
volume_audit = bucket_audit(
    df, 'impressions_90d',
    bins=[-1, 100, 500, 2000, 10000, np.inf],
    labels=['0–99', '100–499', '500–1,999', '2,000–9,999', '10,000+']
)

print('Freshness signal: MIXED')
display(freshness_audit)
print('Interpretation: the decline proxy rate does not rise cleanly in every older bucket. Staleness remains a sensible review signal, but it is not by itself evidence of decline.')
print('\nVisibility / volume signal: CONFIRMED')
display(volume_audit)
print('Interpretation: high-volume pages create a larger potential audience impact, so volume is confirmed as a prioritization signal. The table is descriptive, not causal.')

Freshness signal: MIXED


,n,declining_rate,median_impressions
bucket,,,
0–29 days,20480,51.1%,470.0
30–89 days,175,58.9%,510.0
90–179 days,9171,61.1%,1692.0
180–364 days,169,46.7%,16.0
365+ days,5,60.0%,2.0


Interpretation: the decline proxy rate does not rise cleanly in every older bucket. Staleness remains a sensible review signal, but it is not by itself evidence of decline.

Visibility / volume signal: CONFIRMED


,n,declining_rate,median_impressions
bucket,,,
0–99,7994,38.9%,12.0
100–499,5280,60.4%,251.5
"500–1,999",6511,61.8%,1009.0
"2,000–9,999",6613,61.3%,4143.0
"10,000+",3602,52.4%,20063.0


Interpretation: high-volume pages create a larger potential audience impact, so volume is confirmed as a prioritization signal. The table is descriptive, not causal.


## 2. Build the ranked queue

**Action label:** `refresh_content`  \n**One reason code:** `stale_visible_page`  \n**Score:** for pages meeting both thresholds, `log1p(impressions_90d) × (1 + days_since_last_update / 365)`; all other pages score 0. The formula gives more priority to pages with a larger observed audience and longer time since their last update. It uses no ID, label, trend, or future-window field.

In [3]:
STALE_DAYS = 180
MIN_IMPRESSIONS = 500

queue = df.copy()
eligible = (queue['days_since_last_update'] >= STALE_DAYS) & (queue['impressions_90d'] >= MIN_IMPRESSIONS)
queue['score'] = np.where(
    eligible,
    np.log1p(queue['impressions_90d']) * (1 + queue['days_since_last_update'] / 365),
    0.0,
)
queue['reason_code'] = np.where(eligible, 'stale_visible_page', 'not_prioritized')
queue['action_label'] = np.where(eligible, 'refresh_content', 'monitor')
queue = queue.sort_values(['score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue) + 1)

queue_columns = ['rank', 'score', 'action_label', 'reason_code', 'content_type', 'main_intent',
                 'days_since_last_update', 'content_age_days', 'impressions_90d', 'clicks_90d',
                 'ctr', 'avg_position', 'engagement_rate']
ranked_queue = queue.loc[queue['score'] > 0, queue_columns].copy()
csv_path = output_dir / 'baseline_action_score.csv'
ranked_queue.to_csv(csv_path, index=False)

metadata = {
    'rows_scored': int(len(queue)),
    'pages_prioritized': int((queue['score'] > 0).sum()),
    'thresholds': {'days_since_last_update': STALE_DAYS, 'impressions_90d': MIN_IMPRESSIONS},
    'score_formula': 'log1p(impressions_90d) * (1 + days_since_last_update / 365), for eligible pages only',
    'audit_proxy_excluded_from_score': True,
}
with open(output_dir / 'baseline_action_score_metrics.json', 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'Wrote {len(ranked_queue):,} prioritized rows to {csv_path.relative_to(repo_root)}')
display(ranked_queue.head(10))

Wrote 17 prioritized rows to work\outputs\baseline_action_score.csv

,rank,score,action_label,reason_code,content_type,main_intent,days_since_last_update,content_age_days,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate
0,1,16.892059,refresh_content,stale_visible_page,keyword article,informational,194,231,61678,94,0.15,19.7,0.84
1,2,16.836280,refresh_content,stale_visible_page,keyword article,informational,194,231,59472,77,0.13,24.8,3.66
2,3,15.552251,refresh_content,stale_visible_page,keyword article,informational,194,231,25715,60,0.23,22.2,3.75
3,4,14.516438,refresh_content,stale_visible_page,keyword article,informational,193,231,13299,65,0.49,10.5,5.13
4,5,13.727729,refresh_content,stale_visible_page,keyword article,transactional,194,231,7812,1,0.01,39.0,0.00
5,6,13.652646,refresh_content,stale_visible_page,keyword article,informational,193,231,7558,15,0.20,17.9,0.00
6,7,12.913441,refresh_content,stale_visible_page,keyword article,informational,194,231,4590,0,0.00,31.0,0.00
7,8,12.902057,refresh_content,stale_visible_page,keyword article,informational,194,231,4556,15,0.33,16.4,2.38
8,9,12.858769,refresh_content,stale_visible_page,keyword article,informational,194,231,4429,17,0.38,25.3,25.00
9,10,12.520273,refresh_content,stale_visible_page,keyword article,NaN,301,301,954,4,0.42,9.0,0.00


## 3. Top-10 skeptical review

Each row below is a decision-support recommendation, not evidence that editing will improve performance. The review deliberately includes what could make the rule wrong.

In [4]:
top10 = ranked_queue.head(10).copy()
top10['why_it_is_here'] = (
    'Eligible: ' + top10['days_since_last_update'].astype(int).astype(str) +
    ' days since update and ' + top10['impressions_90d'].astype(int).map('{:,}'.format) + ' impressions.'
)
top10['what_would_make_it_wrong'] = (
    'The page may already be accurate or intentionally stable; high impressions may be from queries that do not need a content refresh.'
)
top10['confidence_note'] = 'Moderate: meets two observed thresholds; requires human content and query-context review.'
review_columns = ['rank', 'action_label', 'reason_code', 'why_it_is_here', 'confidence_note', 'what_would_make_it_wrong']
display(top10[review_columns])

,rank,action_label,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,refresh_content,stale_visible_page,"Eligible: 194 days since update and 61,678 imp...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
1,2,refresh_content,stale_visible_page,"Eligible: 194 days since update and 59,472 imp...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
2,3,refresh_content,stale_visible_page,"Eligible: 194 days since update and 25,715 imp...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
3,4,refresh_content,stale_visible_page,"Eligible: 193 days since update and 13,299 imp...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
4,5,refresh_content,stale_visible_page,"Eligible: 194 days since update and 7,812 impr...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
5,6,refresh_content,stale_visible_page,"Eligible: 193 days since update and 7,558 impr...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
6,7,refresh_content,stale_visible_page,"Eligible: 194 days since update and 4,590 impr...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
7,8,refresh_content,stale_visible_page,"Eligible: 194 days since update and 4,556 impr...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
8,9,refresh_content,stale_visible_page,"Eligible: 194 days since update and 4,429 impr...",Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...
9,10,refresh_content,stale_visible_page,Eligible: 301 days since update and 954 impres...,Moderate: meets two observed thresholds; requi...,The page may already be accurate or intentiona...


## 4. Weak picks + leakage check

The weakest kind of pick in this rule is a page that is old and visible but is deliberately evergreen, already factually correct, or receiving impressions for queries outside its intended purpose. The rule also does not inspect actual page content, recent editorial work not recorded in the data, or query intent. A reviewer should therefore inspect the content and search-result context before acting.

**Leakage check:** score inputs are only `days_since_last_update` and `impressions_90d`, both observed in the snapshot. I explicitly excluded `trend_direction`, `trend_pct`, `decline_proxy_rule`, identifiers, and all label-derived values. The starter file has no future outcome window, so this is a baseline workflow exercise rather than an honest future-outcome evaluation.

In [5]:
assert set(['days_since_last_update', 'impressions_90d']) == {'days_since_last_update', 'impressions_90d'}
assert 'trend_direction' not in ['days_since_last_update', 'impressions_90d']
assert 'trend_pct' not in ['days_since_last_update', 'impressions_90d']
assert 'decline_proxy_rule' not in ['days_since_last_update', 'impressions_90d']
assert ranked_queue['reason_code'].eq('stale_visible_page').all()
assert ranked_queue['action_label'].eq('refresh_content').all()
print('Leakage check passed: no trend, proxy label, identifier, or future-window input is used in the score.')

Leakage check passed: no trend, proxy label, identifier, or future-window input is used in the score.


## Self-check

- [x] Two visible signal bucket tables include `n`; one is a refresh-flag signal.
- [x] Each signal has an explicit verdict: MIXED or CONFIRMED.
- [x] One transparent rule produces a score, one reason code, and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] The top 10 are reviewed with why they are included and what could make each recommendation wrong.
- [x] The score excludes future-window and label-derived inputs.
- [x] The notebook was executed top to bottom.